In [0]:
prod_df = spark.read.table("01_bronze_catalog.raw_schema.product")

In [0]:
from pyspark.sql.functions import when,col,lit
prod_df = prod_df.fillna("unknown",subset=["product_name","plan_name","billing_cycle"])
prod_df = prod_df.withColumn("is_active",when(col("is_active")=="Y" , lit(True)).otherwise(lit(False)))


In [0]:
spark.sql("create schema if not exists 02_silver_catalog.transformed_schema")
prod_df.write.mode("overwrite").option("overwriteSchema","true").saveAsTable("02_silver_catalog.transformed_schema.products")

In [0]:
opp_df = spark.read.table("01_bronze_catalog.raw_schema.opportunity")

In [0]:
from pyspark.sql.functions import when,col,datediff
opp_df = opp_df.withColumn("contract_term",when(col("contract_term").isNull(),when(datediff(col("end_date"),col("start_date")) >= 364,"Yearly").otherwise("Monthly")).otherwise(col("contract_term")))
opp_df = opp_df.fillna("unknown",subset=["close_status"])

In [0]:
from pyspark.sql.functions import col, regexp_replace
opp_df = opp_df.withColumn(
    "revenue_amount",
    regexp_replace(col("revenue_amount"), "[^0-9.]", "").cast("double")
)

In [0]:
cust_df = spark.read.table("02_silver_catalog.transformed_schema.customer")
cm_df = spark.read.table("02_silver_catalog.transformed_schema.country_master")
fx_df = spark.read.table("02_silver_catalog.transformed_schema.fx_rate")

from pyspark.sql.functions import col

opp_df = opp_df.join(cust_df, ['customer_id'], "inner")\
                .join(cm_df, ['country_code'], "inner")\
                .join(fx_df, ['currency_code'], "inner")
opp_df = opp_df.withColumn("revenue_amount",col("revenue_amount")*col("fx_rate_to_gbp"))
opp_df = opp_df.drop("currency_code","fx_rate_to_gbp","effective_date","country_code","country_name","customer_name","industry_type","account_created_date","is_active")

In [0]:
opp_df.write.mode("overwrite").option("overwriteSchema","true").saveAsTable("02_silver_catalog.transformed_schema.opportunity")